# AI012 – M2 Testing Notebook: Local Outlier Factor

This notebook validates the saved production M2 checkpoint and compares LOF output against Role C labels if they are available. It is intended for Role F handoff and team-lead review.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
AI_ML_DIR = PROJECT_DIR.parents[1]

sys.path.insert(0, str(PROJECT_DIR / 'src'))

from models.m2 import (
    LOFAnomalyModel,
    build_output,
    configure_logging,
    load_data,
    save_outputs,
)

configure_logging()

DATASET_PATH = AI_ML_DIR / 'datasets' / 'anomaly_detection_hourly_2020_2024.csv'
CHECKPOINT_PATH = PROJECT_DIR / 'checkpoints' / 'm2.pkl'
PREDICTIONS_PATH = PROJECT_DIR / 'data' / 'processed' / 'lof_predictions.csv'
LABELS_PATH = PROJECT_DIR / 'data' / 'processed' / 'anomaly_labels_v1.csv'

print('Project directory:', PROJECT_DIR)
print('Checkpoint path  :', CHECKPOINT_PATH)
print('Predictions path :', PREDICTIONS_PATH)
print('Labels path      :', LABELS_PATH)

Project directory: /Users/chandupreetham/Phoenix/ai-ml/models/ai012-anomaly
Checkpoint path  : /Users/chandupreetham/Phoenix/ai-ml/models/ai012-anomaly/checkpoints/m2.pkl
Predictions path : /Users/chandupreetham/Phoenix/ai-ml/models/ai012-anomaly/data/processed/lof_predictions.csv
Labels path      : /Users/chandupreetham/Phoenix/ai-ml/models/ai012-anomaly/data/processed/anomaly_labels_v1.csv


## 1. Load the Production Checkpoint

If this cell fails saying the checkpoint was trained with `novelty=False`, regenerate `m2.pkl` by running `m2_train.ipynb` first.

In [2]:
model = LOFAnomalyModel.load_checkpoint(CHECKPOINT_PATH)
print('Loaded model with features:')
for column in model.feature_columns:
    print(' -', column)

[INFO] Loaded checkpoint /Users/chandupreetham/Phoenix/ai-ml/models/ai012-anomaly/checkpoints/m2.pkl trained on 169,539 samples with 6.20% anomaly rate.


Loaded model with features:
 - firms_event_count
 - firms_avg_frp
 - firms_avg_confidence
 - urlhaus_event_count
 - urlhaus_online_count
 - region_lat_bin
 - region_lon_bin
 - cyber_intensity
 - hazard_severity
 - cyber_zscore
 - hazard_zscore
 - combined_risk_index
 - temporal_spike


## 2. Load Dataset and Predictions

If `lof_predictions.csv` is missing, this notebook regenerates it from the checkpoint without re-training.

In [3]:
df = load_data(DATASET_PATH)

if PREDICTIONS_PATH.exists():
    predictions = pd.read_csv(PREDICTIONS_PATH)
    print(f'Loaded predictions: {len(predictions):,} rows')
else:
    print('Predictions file not found; generating from checkpoint...')
    flags = model.predict(df)
    scores = model.score(df)
    predictions = build_output(df, flags, scores)
    save_outputs(predictions, PREDICTIONS_PATH)

predictions.head()

[INFO] Loading dataset: /Users/chandupreetham/Phoenix/ai-ml/datasets/anomaly_detection_hourly_2020_2024.csv
[INFO] Loaded 169,539 rows and 45 columns
[INFO] Computing engineered fallback features from raw columns.


Loaded predictions: 169,539 rows


,time_window,region_id,lof_flag,lof_score,lof_severity
0,2020-01-02 03:00:00,lat_-2_lon_28,0,0.9931,normal
1,2020-10-05 03:00:00,lat_-2_lon_28,0,0.9965,normal
2,2020-10-05 04:00:00,lat_-2_lon_28,1,1.6388,low
3,2021-12-31 03:00:00,lat_-2_lon_28,0,0.9973,normal
4,2022-01-01 04:00:00,lat_-2_lon_28,1,7.8593,critical


## 3. Basic Production Checks

In [4]:
checks = {
    'rows_match_dataset': len(predictions) == len(df),
    'has_lof_flag': 'lof_flag' in predictions.columns,
    'has_lof_score': 'lof_score' in predictions.columns,
    'has_lof_severity': 'lof_severity' in predictions.columns,
    'no_null_scores': predictions['lof_score'].notna().all(),
    'valid_flags': set(predictions['lof_flag'].dropna().unique()).issubset({0, 1}),
}

for name, passed in checks.items():
    print(f'{name:<25}: {"PASS" if passed else "FAIL"}')

assert all(checks.values()), 'One or more production checks failed.'

rows_match_dataset       : PASS
has_lof_flag             : PASS
has_lof_score            : PASS
has_lof_severity         : PASS
no_null_scores           : PASS
valid_flags              : PASS


## 4. Evaluate Against Role C Labels, If Available

In [5]:
if LABELS_PATH.exists():
    labels = pd.read_csv(LABELS_PATH)
    if len(labels) != len(predictions):
        raise ValueError(f'Labels and predictions have different lengths: labels={len(labels)}, predictions={len(predictions)}')

    lof_set = set(predictions.index[predictions['lof_flag'] == 1])
    label_set = set(labels.index[labels['anomaly_flag'] == 1])
    both = lof_set & label_set
    only_lof = lof_set - label_set
    only_label = label_set - lof_set

    print('=== LOF vs Role C Label Overlap ===')
    print(f'LOF flagged        : {len(lof_set):>6,} ({len(lof_set)/len(predictions)*100:.2f}%)')
    print(f'Role C flagged     : {len(label_set):>6,} ({len(label_set)/len(labels)*100:.2f}%)')
    print(f'Both flagged       : {len(both):>6,}')
    print(f'Only LOF           : {len(only_lof):>6,}')
    print(f'Only Role C        : {len(only_label):>6,}')
    print(f'LOF overlap rate   : {len(both)/max(len(lof_set), 1)*100:.1f}%')
else:
    labels = None
    both = set()
    print('Role C labels not found; skipping label-overlap evaluation.')

=== LOF vs Role C Label Overlap ===
LOF flagged        : 10,514 (6.20%)
Role C flagged     : 11,419 (6.74%)
Both flagged       :    747
Only LOF           :  9,767
Only Role C        : 10,672
LOF overlap rate   : 7.1%


## 5. Inspect Highest-Scoring Anomalies

In [6]:
anom = predictions[predictions['lof_flag'] == 1].sort_values('lof_score', ascending=False)
anom.head(10)

,time_window,region_id,lof_flag,lof_score,lof_severity
127650,2023-11-07 05:00:00,lat_-7_lon_25,1,40.0366,critical
127641,2021-05-09 16:00:00,lat_-7_lon_25,1,33.4344,critical
115818,2021-12-08 17:00:00,lat_-7_lon_22,1,28.1146,critical
115814,2020-05-14 05:00:00,lat_-7_lon_22,1,25.8393,critical
127659,2024-12-13 05:00:00,lat_-7_lon_25,1,24.8839,critical
127652,2023-12-27 16:00:00,lat_-7_lon_25,1,24.1940,critical
115820,2021-12-11 06:00:00,lat_-7_lon_22,1,18.0938,critical
127640,2020-09-19 16:00:00,lat_-7_lon_25,1,14.1331,critical
130889,2022-05-02 15:00:00,lat_-7_lon_27,1,13.6891,critical
127647,2023-10-11 05:00:00,lat_-7_lon_25,1,13.3047,critical


## 6. Domain Validation

This checks whether LOF anomalies have stronger fire/cyber signals than normal rows.

In [7]:
check_cols = [c for c in ['firms_avg_frp', 'urlhaus_event_count', 'urlhaus_online_count', 'firms_avg_confidence'] if c in df.columns]
anom_rows = predictions['lof_flag'] == 1

if check_cols:
    domain_check = pd.DataFrame({
        'normal_mean': df.loc[~anom_rows, check_cols].mean(numeric_only=True),
        'anomaly_mean': df.loc[anom_rows, check_cols].mean(numeric_only=True),
        'anomaly_max': df.loc[anom_rows, check_cols].max(numeric_only=True),
    })
    display(domain_check.round(3))
else:
    print('Expected domain columns not found; skipping domain validation.')

,normal_mean,anomaly_mean,anomaly_max
firms_avg_frp,6.893,6.871,606.12
urlhaus_event_count,0.048,0.087,21.00
urlhaus_online_count,0.048,0.087,21.00
firms_avg_confidence,52.561,52.251,100.00


## 7. Summary for Role F / Team Lead

In [8]:
print('=== M2 LOF Summary ===')
print('Model             : Local Outlier Factor')
print('Production mode   : novelty=True, supports checkpoint inference')
print(f'n_neighbors       : {model.n_neighbors}')
print(f'contamination     : {model.contamination}')
print(f'Total records     : {len(predictions):,}')
print(f'Flagged anomalies : {predictions["lof_flag"].sum():,} ({predictions["lof_flag"].mean()*100:.2f}%)')
print(f'Score range       : [{predictions["lof_score"].min():.4f}, {predictions["lof_score"].max():.4f}]')
print(f'Checkpoint        : {CHECKPOINT_PATH.relative_to(PROJECT_DIR)}')
print(f'Predictions       : {PREDICTIONS_PATH.relative_to(PROJECT_DIR)}')
print('Why LOF?          : It detects local density outliers, so it complements global detectors like Isolation Forest.')

=== M2 LOF Summary ===
Model             : Local Outlier Factor
Production mode   : novelty=True, supports checkpoint inference
n_neighbors       : 20
contamination     : 0.07
Total records     : 169,539
Flagged anomalies : 10,514 (6.20%)
Score range       : [0.8032, 40.0366]
Checkpoint        : checkpoints/m2.pkl
Predictions       : data/processed/lof_predictions.csv
Why LOF?          : It detects local density outliers, so it complements global detectors like Isolation Forest.
